# 초구 예측 — model_19 (3단계 마감: 위치 파라미터 다시 고르기)

- 3단계 마감 "단계 끝 다시 고르기" (사용자 결정): 위치 정규분포 파라미터를 지금 구조(model_18: 시즌 초·모든 카운트·분산 따로·절대 높이)에서 다시 고름
  - 특히 1단계 위치 파라미터(기준점·당김·반감기)는 틀린 z_norm 가정 위에서 고른 값 (4-39)
- 묶음 4개를 번갈아 고름 (묶음 안 파라미터는 함께): 평균 당김 (반감기·k_type·k_hand·시즌 초 배수) / 기준점 반감기 / 모든 카운트 (δ 반감·k_b) / 분산 따로 (분산 반감·k_var·k0)
  - 묶음마다 거친 격자 -> 범위 끝이면 넓힘 -> 1등 주변 세밀한 격자, 한 바퀴 돌고 바뀐 게 있으면 한 바퀴 더
  - 기준: 2019 실제 구종 기준 24칸 log loss (상황 보정 전 정규분포, model_11–13과 같은 단계)
- 상황 보정(월 단위, LightGBM 파라미터 그대로)은 새 위치를 출발점으로 다시 학습, 구종은 그대로
- 채택 기준 (다): 24칸 + H/S/C log loss도 부트스트랩에서 나빠지지 않아야 함
- 주의: 파라미터 10개를 2019로 고름 -> 2019 이득은 부풀려질 수 있음 (model_15·17 교훈), 2020으로 확인
- 비교 대상: model_18

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_19"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 모델

In [2]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 구종·상황 보정은 model_18 값, 위치 정규분포 출발 값은 model_18 -> 3-1에서 다시 고른 값으로 바꿈

In [3]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 정규분포: model_13 값 (고정)
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k
LOC_SEASON_F = 0.707  # 시즌 초 배수
LOC_AC = (None, 12.5, 17.3)  # 모든 카운트 위치 성향 (잔차 기준 반감 None = 누적, δ 반감, δ 당김 k)
LOC_VAR = (200.0, 9.74, False, 316.0)  # 분산 따로 (다른 카운트 퍼짐 반감, k_var, unc, 투수 안 분산 당김 k0)
# ② 위치 상황 보정 (model_14): 24칸 LightGBM, 출발점 = model_13 위치 log 확률
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # 상황 변수 (0-3 묶음 4)
LOC_FEAT = CLASSES + ["same_side"] + LOC_SIT  # 구종 원-핫·타자 손 + 상황
LOC_TRAIN_FIRST = 2018  # 위치 보정 학습 첫 해 (2017은 model_13 위치 없음)
LOC_LGB, LOC_ROUNDS = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}, 168  # model_14 상황 보정 LightGBM (고정)
LOC_RETRAIN = "month"  # 상황 보정 재학습 주기 (model_16): 그 달 첫 경기 전날까지 데이터로 다시 학습
LOC_HEIGHT = "abs"  # 위치 높이 표현 (model_18): 절대 높이 cm (지금까지는 존 비율 z_norm)
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [4]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 구종 재현 확인용 (구종은 model_09 그대로)
m13 = pd.read_parquet("results/model_13/per_pitch.parquet")  # 위치 정규분포 재현 확인용
m14 = pd.read_parquet("results/model_14/per_pitch.parquet")  # 참고
m16 = pd.read_parquet("results/model_16/per_pitch.parquet")  # 참고
m18 = pd.read_parquet("results/model_18/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [5]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [6]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_19    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_19    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: 파라미터 다시 고른 절대 높이 정규분포 + 상황 보정 (월 단위)

### 3-0. 위치 함수 (model_18와 같음, 절대 높이)

In [8]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = gauss_abs(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)  # 절대 높이 (model_18)
    return zone_probs(g, qt, cov), g


# 높이 표현 (model_18): 절대 높이 cm, 상수는 2017–18 값 (2019·2020 모두 같은 상수, 미래 정보 없음)
past_h = fp["GYEAR"] < fc.VAL_YEAR
H_BAR = float((fp.loc[past_h, "TOP_SZ_CM"] - fp.loc[past_h, "BOTTOM_SZ_CM"]).mean())  # 평균 존 높이
BOT_BAR = float(fp.loc[past_h, "BOTTOM_SZ_CM"].mean())  # 평균 아랫선
fp["zn"] = (fp["CROSS_PLATE_Z_CM"] - BOT_BAR) / H_BAR  # 높이 열을 절대 높이 표현으로 (구종은 이 열을 안 씀)
fp["zz"], fp["xz"] = fp["zn"] ** 2, fp["x"] * fp["zn"]
MO["zn"] = (MO["CROSS_PLATE_Z_CM"] - BOT_BAR) / H_BAR  # 다른 카운트 공도 같은 표현
for cache in (fc._TABLES, _MO_RES, _MO_MOM, _WT):
    cache.clear()  # 높이 열이 바뀌어 누적표를 새로 (prior_table 캐시는 id(data) 키라 같은 객체면 옛 표를 씀)

def gauss_abs(M, q):  # 모멘트 (x, zn, xx, zz, xz) -> 2차원 정규분포, 높이는 절대 높이로 되돌림 (타자 존 무관)
    mx, mz, xx, zz, xz = M[:, :5].T
    vx = np.maximum(xx - mx ** 2, 4.0)
    vz = np.maximum(zz - mz ** 2, 1e-3) * H_BAR ** 2
    c = np.clip((xz - mx * mz) * H_BAR, -0.99 * np.sqrt(vx * vz), 0.99 * np.sqrt(vx * vz))
    return mx, BOT_BAR + mz * H_BAR, vx, vz, c

### 3-1. 위치 정규분포 파라미터 다시 고르기 (2019)

- 대조 = model_18 값 (기준점 반감 173 · 반감 35.4 · k_type 1000 · k_hand 30 · 시즌 초 0.707 · 모든 카운트 12.5 · 17.3 · 분산 200 · 9.74 · k0 316), 2019 2.87214 재현 확인
- 묶음 4개를 번갈아: 묶음마다 다른 묶음은 그때 값으로 고정
- 범위 끝 넓히기 (시즌 초 배수는 1 위로 안 넓힘), 세밀한 격자는 평균 당김 묶음은 반감기·k_hand만 (나머지 둘은 거친 값)

In [9]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN -> None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

def game_ci(d, gid, B=1000):  # 경기 단위 부트스트랩 95% 구간 (같은 경기 공은 독립이 아님)
    s = pd.DataFrame({"g": gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])





yz9 = q_val["zone"].map(fc.ZONE_IDX).to_numpy()
i9 = np.arange(len(q_val))
gid9 = q_val["gameID"].to_numpy()
_REF, _SN, _AC, _OV = {}, {}, {}, {}  # 2019 lookup 캐시 (prior_table은 캐시 안 함)

def ll_p(ref_hl, half_life, k_type, k_hand, season_f, hl_b, k_b, hl_var, k_var, k0):  # 2019 공별 24칸 log loss (실제 구종, 상황 보정 전)
    if ref_hl not in _REF:
        _REF[ref_hl] = loc_league(q_val, fc.VAL_YEAR, "decay", ref_hl)
    if (half_life, season_f) not in _SN:
        _SN[(half_life, season_f)] = loc_sn(q_val, half_life, season_f, cache=False)
    if (hl_b, season_f) not in _AC:
        _AC[(hl_b, season_f)] = ac_sn(q_val, None, hl_b, season_f, cache=False)
    if (hl_var, season_f, k0) not in _OV:
        _OV[(hl_var, season_f, k0)] = own_var(q_val, fc.VAL_YEAR, None, hl_var, season_f, k0, cache=False)
    (S1, N1), (S2, N2) = _SN[(half_life, season_f)]
    S, N = _AC[(hl_b, season_f)]
    d = S / (N + k_b)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, shift_mean(_REF[ref_hl], d[:, 0], d[:, 1]), k_type), k_hand)
    M = apply_var(M, *_OV[(hl_var, season_f, k0)], k_var, False, N2, k_hand)
    P = fc.zone_from_gauss_cov(gauss_abs(M, q_val), q_val, ZONE_M)
    return -np.log(np.clip(P[i9, yz9], 1e-12, None))

CUR = dict(ref_hl=173.0, half_life=35.4, k_type=1000.0, k_hand=30.0, season_f=0.707, hl_b=12.5, k_b=17.3, hl_var=200.0, k_var=9.74, k0=316.0)  # model_18
LL22 = ll_p(**CUR)
assert abs(LL22.mean() - 2.87214) < 2e-5, LL22.mean()
print(f"model_18 위치 (상황 보정 전) 2019 24칸 log loss {LL22.mean():.5f} (0장 model_18 값과 같음)")

GROUPS = {"평균 당김": {"half_life": [25.0, 35.4, 50.0], "k_type": [300.0, 1000.0, 3000.0], "k_hand": [20.0, 30.0, 45.0], "season_f": [0.5, 0.707, 1.0]},
          "기준점": {"ref_hl": [100.0, 173.0, 300.0]},
          "모든 카운트": {"hl_b": [8.84, 12.5, 17.7], "k_b": [10.0, 17.3, 30.0]},
          "분산 따로": {"hl_var": [141.0, 200.0, 283.0], "k_var": [5.48, 9.74, 17.3], "k0": [100.0, 316.0, 1000.0]}}
FINE = {"평균 당김": ["half_life", "k_hand"], "기준점": ["ref_hl"], "모든 카운트": ["hl_b", "k_b"], "분산 따로": ["hl_var", "k_var", "k0"]}

def score_with(fixed):
    return lambda **p: {"logloss": ll_p(**{**fixed, **p}).mean()}

def select_group(name, cur):  # 묶음 하나 다시 고르기 -> 새 값 dict, 기록
    grid = {k: list(v) for k, v in GROUPS[name].items()}
    for k in grid:  # 지금 값이 격자에 없으면 넣음 (대조가 꼭 들어가게)
        if cur[k] not in grid[k]:
            grid[k] = sorted(grid[k] + [cur[k]])
    fixed = {k: v for k, v in cur.items() if k not in grid}
    sc = score_with(fixed)
    res = grid_search(sc, grid)
    tried = {k: list(v) for k, v in grid.items()}
    for rnd in range(3):  # 범위 끝이면 넓힘
        b = best_of(res, grid)
        add = {}
        for k, v in b.items():
            if v == min(tried[k]):
                add[k] = [round(v / 2, 3)]
            elif v == max(tried[k]) and not (k == "season_f" and v >= 1.0):
                add[k] = [min(round(v * 2, 3), 1.0) if k == "season_f" else round(v * 2, 3)]
        add = {k: v for k, v in add.items() if v[0] not in tried[k]}
        if not add:
            break
        for k, v in add.items():
            tried[k] += v
            res = pd.concat([res, grid_search(sc, {**{kk: [b[kk]] for kk in grid}, k: v})])
        res = res.drop_duplicates(list(grid)).sort_values("logloss").reset_index(drop=True)
    b = best_of(res, grid)
    fg = {k: ([v for v in finer(tried[k], b[k]) if not (k == "season_f" and v > 1.0)] if k in FINE[name] else [b[k]]) for k in grid}
    res = pd.concat([res, grid_search(sc, fg)]).drop_duplicates(list(grid)).sort_values("logloss").reset_index(drop=True)
    b = best_of(res, grid)
    for k in FINE[name]:
        tried[k] = sorted(set(tried[k]) | set(fg[k]))
    edge = [k for k in grid if b[k] in (min(tried[k]), max(tried[k])) and not (k == "season_f" and b[k] >= 1.0)]
    before = res[np.logical_and.reduce([res[k] == cur[k] for k in grid])].logloss.iloc[0]
    return b, {"묶음": name, "전": {k: cur[k] for k in grid}, "후": b, "전 ll": before, "후 ll": res.logloss.iloc[0], "범위 끝": edge or "없음"}

LOG = []
cur = dict(CUR)
for rnd in range(2):  # 한 바퀴 돌고 바뀐 게 있으면 한 바퀴 더
    changed = False
    for name in GROUPS:
        b, rec = select_group(name, cur)
        rec["바퀴"] = rnd + 1
        LOG.append(rec)
        if any(b[k] != cur[k] for k in b):
            changed = True
        cur.update(b)
    if not changed:
        break
display(pd.DataFrame(LOG)[["바퀴", "묶음", "전", "후", "전 ll", "후 ll", "범위 끝"]])
LOC_NEW = dict(cur)
LLN = ll_p(**LOC_NEW)
dd = LL22 - LLN
s = pd.DataFrame({"g": gid9, "d": dd}).groupby("g")["d"].agg(["sum", "count"])
ii = np.random.default_rng(0).integers(0, len(s), (1000, len(s)))
lo, hi = np.percentile(s["sum"].to_numpy()[ii].sum(1) / s["count"].to_numpy()[ii].sum(1), [2.5, 97.5])
print("model_18 값:", CUR)
print("새 값:", LOC_NEW)
print(f"2019 24칸 (상황 보정 전): model_18 {LL22.mean():.5f} -> 새 값 {LLN.mean():.5f}, 이득 ×1000 {dd.mean() * 1000:.2f} (95% {lo * 1000:.2f} ~ {hi * 1000:.2f})")

model_18 위치 (상황 보정 전) 2019 24칸 log loss 2.87214 (0장 model_18 값과 같음)


,바퀴,묶음,전,후,전 ll,후 ll,범위 끝
0,1,평균 당김,"{'half_life': 35.4, 'k_type': 1000.0, 'k_hand'...","{'half_life': 50.0, 'k_type': 24000.0, 'k_hand...",2.872141,2.871923,[k_type]
1,1,기준점,{'ref_hl': 173.0},{'ref_hl': 424.0},2.871923,2.871781,없음
2,1,모든 카운트,"{'hl_b': 12.5, 'k_b': 17.3}","{'hl_b': 10.5, 'k_b': 17.3}",2.871781,2.871772,없음
3,1,분산 따로,"{'hl_var': 200.0, 'k_var': 9.74, 'k0': 316.0}","{'hl_var': 200.0, 'k_var': 13.0, 'k0': 562.0}",2.871772,2.871747,없음
4,2,평균 당김,"{'half_life': 50.0, 'k_type': 24000.0, 'k_hand...","{'half_life': 42.1, 'k_type': 192000.0, 'k_han...",2.871747,2.871737,[k_type]
5,2,기준점,{'ref_hl': 424.0},{'ref_hl': 424.0},2.871737,2.871737,없음
6,2,모든 카운트,"{'hl_b': 10.5, 'k_b': 17.3}","{'hl_b': 9.63, 'k_b': 17.3}",2.871737,2.871735,없음
7,2,분산 따로,"{'hl_var': 200.0, 'k_var': 13.0, 'k0': 562.0}","{'hl_var': 200.0, 'k_var': 11.3, 'k0': 421.0}",2.871735,2.871720,없음


model_18 값: {'ref_hl': 173.0, 'half_life': 35.4, 'k_type': 1000.0, 'k_hand': 30.0, 'season_f': 0.707, 'hl_b': 12.5, 'k_b': 17.3, 'hl_var': 200.0, 'k_var': 9.74, 'k0': 316.0}
새 값: {'ref_hl': 424.0, 'half_life': 42.1, 'k_type': 192000.0, 'k_hand': 24.5, 'season_f': 0.707, 'hl_b': 9.63, 'k_b': 17.3, 'hl_var': 200.0, 'k_var': 11.3, 'k0': 421.0}
2019 24칸 (상황 보정 전): model_18 2.87214 -> 새 값 2.87172, 이득 ×1000 0.42 (95% -0.13 ~ 1.02)


#### 선택: 두 바퀴 돌았지만 이득은 잡음 수준 (2019 +0.42 ×1000, 95% −0.13 ~ +1.02)
| 묶음 | model_18 | 다시 고른 값 | 2019 24칸 (상황 보정 전) |
| --- | --- | --- | --- |
| 평균 당김 | 반감 35.4 · k_type 1000 · k_hand 30 · 시즌 초 0.707 | 42.1 · **192000** · 24.5 · 0.707 | 2.87214 -> 2.87192 (1바퀴) |
| 기준점 | 173 | 424 | -> 2.87178 |
| 모든 카운트 | 12.5 · 17.3 | 9.63 · 17.3 | -> 2.87177 |
| 분산 따로 | 200 · 9.74 · 316 | 200 · 11.3 · 421 | -> 2.87172 |

- 전체 2.87214 -> 2.87172 (−0.0004), 95% 구간이 0 포함 -> 지금 값이 이미 거의 최적, 절대 높이로 바꿔도 1단계·위치 1단계 보완 파라미터는 그대로 맞음
- k_type이 넓힐 때마다 끝으로 감 (192000 = 사실상 무한대): 투수×구종(손 구분 없음) 단계를 안 쓰고 기준점에서 바로 투수×구종×손으로 가는 게 같거나 조금 나음 -> 지금도 1000이라 거의 안 쓰는 단계

In [10]:
LOC_REF_HL, LOC_HL, K_TYPE, K_HAND, LOC_SEASON_F = LOC_NEW["ref_hl"], LOC_NEW["half_life"], LOC_NEW["k_type"], LOC_NEW["k_hand"], LOC_NEW["season_f"]
LOC_AC = (None, LOC_NEW["hl_b"], LOC_NEW["k_b"])
LOC_VAR = (LOC_NEW["hl_var"], LOC_NEW["k_var"], False, LOC_NEW["k0"])
LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)  # 실제 구종 기준 위치 24칸 확률 (2018–2020, 새 값)
for yr in range(LOC_TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]
ll_ = -np.log(np.clip(PZ_TRUE[val][i9, yz9], 1e-12, None))
assert np.allclose(ll_, LLN), "3-1 값과 다름"
print(f"위치 (새 값, 상황 보정 전) 2019 24칸 {ll_.mean():.5f} = 3-1 (재현 확인), 2018 계산된 공 {int(np.isfinite(PZ_TRUE[fp['GYEAR'].eq(2018).to_numpy()]).all(1).sum())}")

위치 (새 값, 상황 보정 전) 2019 24칸 2.87172 = 3-1 (재현 확인), 2018 계산된 공 12395


### 3-2. 상황 보정 월 단위 재학습 (model_16과 같음, 출발점만 새 위치)

In [11]:
from scipy.special import log_softmax
for fn in (fc.situation_features, fc.opponent_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(LOC_TRAIN_FIRST)], n=150)
    assert bad.empty, bad
SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]]], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)  # 득점권인데 1루 비었음
XLOC = XLOC[LOC_FEAT]
NZ = len(ZONES)
INIT_L = np.log(np.clip(PZ_TRUE, 1e-6, None))
INIT_L -= INIT_L.mean(1, keepdims=True)  # 출발점
yr_l = fp["GYEAR"].to_numpy()
BASE_L = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)


DATE_L = fp["game_date"].to_numpy()
MON_L = np.clip(fp["game_date"].dt.month.to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 상황 보정 LightGBM (model_14 파라미터, 트리 168)
    return lgb.train({**BASE_L, **LOC_LGB}, lgb.Dataset(XLOC.iloc[tr], y_zone[tr], init_score=INIT_L[tr]), LOC_ROUNDS)

BST_L, plan = {}, []  # (연도, 달) -> 그 달 예측용 모델
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_l == yr)
    for mon in np.unique(MON_L[iy]):
        te = iy[MON_L[iy] == mon]
        start = DATE_L[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_l >= LOC_TRAIN_FIRST) & (DATE_L < start))
        assert DATE_L[tr].max() < start  # 미래 정보 없음
        BST_L[(yr, mon)] = fit_rows(tr)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_l[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE_L[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))

def loc_correct(P, q, year, t=None):  # model_13 24칸 확률 P (n, 24) -> 상황 보정 확률 (공마다 그 달 모델), t = 가정한 구종 (None이면 q의 실제 구종)
    X = XLOC.loc[q.index].copy()
    if t is not None:
        X.loc[:, CLASSES] = np.tile((np.array(CLASSES) == t).astype(float), (len(X), 1))  # 구종 원-핫을 t로
    init = np.log(np.clip(P, 1e-6, None))
    init -= init.mean(1, keepdims=True)
    mon = MON_L[fp.index.get_indexer(q.index)]
    out = np.empty_like(P)
    for mo in np.unique(mon):
        k = mon == mo
        out[k] = np.exp(log_softmax(BST_L[(year, mo)].predict(X[k], raw_score=True) + init[k], axis=1))
    return out



미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  12395           0  2018-10-14  2019-03-23
     5   2403  14963        2568  2019-04-30  2019-05-01
     6   2493  17366        4971  2019-05-31  2019-06-01
     7   1791  19859        7464  2019-06-30  2019-07-02
     8   2362  21650        9255  2019-07-31  2019-08-01
     9   1841  24012       11617  2019-08-31  2019-09-01
2020 5   2264  25853           0  2019-10-01  2020-05-05
     6   2379  28117        2264  2020-05-31  2020-06-02
     7   2233  30496        4643  2020-06-30  2020-07-01
     8   1985  32729        6876  2020-07-31  2020-08-01
     9   3754  34714        8861  2020-08-30  2020-09-01

### 3-3. 진단 (실제 구종 기준)

- model_18 대비 이득: 달별·주자·타자 존 높이, 퍼짐 보정

In [12]:
def auc(p, o):  # Mann-Whitney AUC (같은 값은 평균 순위)
    r = pd.Series(p).rank().to_numpy()
    n1, n0 = o.sum(), len(o) - o.sum()
    return (r[o == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def runner3(q):
    return np.select([(q["base2"] + q["base3"]) > 0, q["base1"] > 0], ["득점권", "1루만"], "없음")

rows, cal = [], []
for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:
    i, yz = np.arange(len(q)), y_zone[msk]
    P22 = loc_correct(PZ_TRUE[msk], q, yr)
    a20 = fc.align(m18, fp, yr)  # 비교 대상 model_18 (변수 이름은 그대로)
    ll20, ll22 = a20["ll_zone_true"].to_numpy(), -np.log(np.clip(P22[i, yz], 1e-12, None))
    hz = (q["TOP_SZ_CM"] - q["BOTTOM_SZ_CM"]).to_numpy()
    h3 = np.asarray(pd.cut(hz, [0, 54.3, 57.2, 999], labels=["존 낮은 타자", "중간", "존 높은 타자"]))  # 0장 3등분 경계
    oc = (fc.ZONE_GROUP[yz] == "C").astype(float)
    d = pd.DataFrame({"월": MON_L[msk], "주자": runner3(SIT_L.loc[q.index]), "존 높이": h3, "이득": (ll20 - ll22) * 1000})
    for var in ["월", "주자", "존 높이"]:
        for v, g in d.groupby(var):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "이득 ×1000 (model_18 대비)": g["이득"].mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "이득 ×1000 (model_18 대비)": d["이득"].mean(),
                 "존 밖 AUC": auc(P22[:, fc.ZONE_GROUP == "C"].sum(1), oc), "H/S/C 정확도": ((P22 @ fc.GROUP2).argmax(1) == pd.Series(fc.ZONE_GROUP[yz]).map({"H": 0, "S": 1, "C": 2}).to_numpy()).mean()})
    mx, mz, vx, vz, c = loc_given_type(q, yr, q["pitch_type"].to_numpy(), **LOC17)[1]
    dx, dz = q["x"].to_numpy() - mx, q["CROSS_PLATE_Z_CM"].to_numpy() - mz
    d2 = (vz * dx ** 2 - 2 * c * dx * dz + vx * dz ** 2) / (vx * vz - c ** 2)
    for lab in ["존 낮은 타자", "중간", "존 높은 타자", "전체"]:
        k = (h3 == lab) if lab != "전체" else np.ones(len(q), bool)
        cal.append({"연도": yr, "존 높이": lab, "평균 ez": (dz / np.sqrt(vz))[k].mean(), "SD ez": (dz / np.sqrt(vz))[k].std(), "80% 타원": (d2[k] <= -2 * np.log(0.2)).mean()})
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(4))
print("퍼짐 보정 (절대 높이, 실제 구종, 상황 보정 전 정규분포)")
display(pd.DataFrame(cal).set_index(["존 높이", "연도"]).unstack("연도").round(3))

공          이득 ×1000 (model_18 대비)         존 밖 AUC         H/S/C 정확도        
연도               2019     2020                   2019    2020    2019    2020      2019    2020
변수   값                                                                                         
월    4         2568.0      NaN                -0.0197     NaN     NaN     NaN       NaN     NaN
     5         2403.0   2264.0                 1.4780 -0.8587     NaN     NaN       NaN     NaN
     6         2493.0   2379.0                 0.9266 -0.2381     NaN     NaN       NaN     NaN
     7         1791.0   2233.0                 1.4869  0.6188     NaN     NaN       NaN     NaN
     8         2362.0   1985.0                 0.1848  0.6699     NaN     NaN       NaN     NaN
     9         1841.0   3754.0                -0.5742  0.8466     NaN     NaN       NaN     NaN
전체            13458.0  12615.0                 0.5836  0.2679  0.5641  0.5601    0.4096  0.4094
존 높이 존 낮은 타자   4638.0   5407.0                 0.1113  0.3736     NaN     NaN       NaN     NaN
     존 높은 타자   3711.0   1377.0                 1.2504  0.0696     NaN     NaN       NaN     NaN
     중간        5109.0   5831.0                 0.5279  0.2167     NaN     NaN       NaN     NaN
주자   1루만       2886.0   2723.0                 1.0356  0.7022     NaN     NaN       NaN     NaN
     득점권       3346.0   3136.0                 1.0932 -0.6504     NaN     NaN       NaN     NaN
     없음        7226.0   6756.0                 0.1670  0.5190     NaN     NaN       NaN     NaN

퍼짐 보정 (절대 높이, 실제 구종, 상황 보정 전 정규분포)


평균 ez         SD ez        80% 타원       
연도        2019   2020   2019   2020   2019   2020
존 높이                                             
전체       0.032 -0.014  0.999  1.005  0.815  0.807
존 낮은 타자 -0.004  0.005  0.991  1.014  0.820  0.810
존 높은 타자  0.030 -0.069  1.008  0.982  0.798  0.808
중간       0.066 -0.020  0.999  1.000  0.823  0.804

#### 🔍 3-3 결과: model_18와 거의 같음
- model_18 대비 이득 ×1000 (실제 구종, 상황 보정 후): 2019 +0.58, 2020 +0.27
- 달별 2020: −0.9 · −0.2 · +0.6 · +0.7 · +0.8 / 주자별 2020 득점권 −0.7 -> 방향이 섞임
- 퍼짐 보정은 model_18와 같음 (80% 타원 0.815 / 0.807)

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_12과 같음, 위치만 3-1에서 고른 값 (분산 따로 포함)

In [13]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, LOC_VAR)
        P = loc_correct(P, q, year, t)  # 상황 보정 (model_14), 정규분포 g는 model_13 그대로
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [14]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종은 model_18와 같아야 함 (위치는 24칸·좌표 모두 바뀜)
chk = pp.merge(m18, on=fc.KEYS, suffixes=("", "_18"))
same = ["ll_type"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_18"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_18와 다른 열: {bad}"
print("구종 = model_18 (재현 확인), 위치(24칸·좌표)만 바뀜")

(26073, 48)
구종 = model_18 (재현 확인), 위치(24칸·좌표)만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_18와 나란히 / 경기 단위 부트스트랩 (model_19 − model_18, 2020), H/S/C(`ll_hsc`)도 나빠지지 않아야 함
- 2019는 파라미터를 고른 해라 부풀려질 수 있음 -> 2020으로 판단

In [15]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m18_sum = eval_years(m18)
display(pd.concat({MODEL: summary, "model_18": m18_sum, "차이 (model_18 대비)": summary - m18_sum}, axis=1).round(4))

model_19          model_18          차이 (model_18 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9277   2.9265   2.9278   2.9266          -0.0001 -0.0001
24칸 top1              0.1384   0.1374   0.1384   0.1371           0.0000  0.0003
24칸 top3              0.3064   0.3054   0.3041   0.3066           0.0024 -0.0013
H/S/C logloss         1.0722   1.0729   1.0723   1.0728          -0.0001  0.0000
H/S/C 정확도             0.4025   0.4036   0.4019   0.4024           0.0006  0.0013
24칸 logloss (실제 구종)   2.8672   2.8700   2.8678   2.8703          -0.0006 -0.0003
구종×칸 logloss          3.9285   3.9261   3.9291   3.9264          -0.0006 -0.0003
좌표 평균거리 cm           32.3946  31.6318  32.3944  31.6319           0.0002 -0.0001
좌표 NLL                9.2951   9.2490   9.2949   9.2493           0.0002 -0.0003

In [16]:
display(fc.bootstrap_diff(fc.pitch_scores(m18, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    a20, a22 = fc.align(m18, fp, yr), fc.align(pp, fp, yr)  # a20 = 비교 대상 model_18
    nk = a20["pitch_type"].ne("KNUC").to_numpy()
    d_true = a22["ll_zone_true"].to_numpy() - a20["ll_zone_true"].to_numpy()
    print(f"{yr} 24칸 실제 구종 차이 (model_19 − model_18): 전체 {d_true.mean():+.5f}, 너클 뺌 {d_true[nk].mean():+.5f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0000,0.0000,0.0000,차이 불확실
ll_zone,-0.0001,-0.0006,0.0003,차이 불확실
ll_hsc,0.0000,-0.0001,0.0001,차이 불확실
ll_joint,-0.0003,-0.0008,0.0003,차이 불확실
nll_coord,-0.0003,-0.0008,0.0001,차이 불확실
dist,-0.0001,-0.0072,0.0072,차이 불확실


2019 24칸 실제 구종 차이 (model_19 − model_18): 전체 -0.00058, 너클 뺌 -0.00058
2020 24칸 실제 구종 차이 (model_19 − model_18): 전체 -0.00027, 너클 뺌 -0.00029


### 🥇1등🥇
#### 다시 골라도 차이는 잡음 수준 (2019 상황 보정 전 −0.0004, 2020 실전 −0.0001) -> 채택 안 함, model_18 값 유지
- 부트스트랩 (2020, model_19 − model_18): 24칸·H/S/C 포함 7개 점수 모두 차이 불확실
- 평평한 구간 규칙: 파라미터 10개를 2019로 골랐고 k_type은 범위 끝(192000)으로 감 -> 지금 값 유지
- 결론: 절대 높이로 바꿔도 1단계·위치 1단계 보완 파라미터는 그대로 맞음 (2단계 마감 "단계 끝 다시 고르기"와 같은 결론)
- **채택 안 함 -> model_18가 계속 비교 출발점** (사용자 결정 2026-11-11)

## 6. 저장

- `results/model_19/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [17]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_19 ['per_pitch.parquet', 'summary.csv']
